### 1. Variáveis instrumentais e mínimos quadrados em dois estágios

O conjunto de códigos abaixo replica alguns exemplos práticos do Capítulo 15, intitulado **"Estimação de variáveis instrumentais e mínimos quadrados em dois estágios"**, do livro *Introdução à Econometria*, tradução da 6º edição Norte Americana, do autor Jeffrey M. Wooldridge. Este capítulo aborda a estimativa de modelos econométricos com o método de variáveis instrumentais, destacando o uso de mínimos quadrados em dois estágios para corrigir problemas de endogeneidade e obter estimativas consistentes. O livro, em formato digital e físico, pode ser adquirido no seguinte site: [amazon.com.br](https://www.amazon.com.br/Introdu%C3%A7%C3%A3o-%C3%A0-econometria-abordagem-moderna/dp/8522125643?ref_=ast_author_dp&dib=eyJ2IjoiMSJ9.xpLzGp_gZEtZm2kFmBCS0if6iw5aeB-m4qX1SKcoLrzW2xj1vkpOLp1CDoM7qPymZwglN1USzM69oUsMlZMlc8hm2kJEe2SLc4ft_i_GErH5YT2BZepNJLVH2D1cAGzU3WfbeyFYrMs4djQeG0V0I3MA2aVgo4IqjdGSKZzvuHkfd18qWqoBjSYovRqPNAU0sI9kwF6RwGG2EatJQ0LuUUOSw_OTKr29SeqG01Y7cds.zp4BJt5PaAnnWcFufRwtdk6xs-2WFdGq2UM5uH3Xrbc&dib_tag=AUTHOR).


### 2. Bibliotecas

In [1]:
# Manipulação dos dados
import pandas as pd
import numpy as np

# Estatística
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

# Dados
import wooldridge as woo

# Configurações
import warnings

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
warnings.filterwarnings("ignore")

### Exemplo 15.1 - Estimação do retorno da educação para mulheres casadas

In [3]:
# Importando os dados
mroz = woo.dataWoo('mroz')
mroz.head()

,inlf,hours,kidslt6,kidsge6,age,educ,wage,repwage,hushrs,husage,huseduc,huswage,faminc,mtr,motheduc,fatheduc,unem,city,exper,nwifeinc,lwage,expersq
0,1,1610,1,0,32,12,3.3540,2.65,2708,34,12,4.0288,16310.0,0.7215,12,7,5.0,0,14,10.910060,1.210154,196
1,1,1656,0,2,30,12,1.3889,2.65,2310,30,9,8.4416,21800.0,0.6615,7,7,11.0,1,5,19.499981,0.328512,25
2,1,1980,1,3,35,12,4.5455,4.04,3072,40,12,3.5807,21040.0,0.6915,12,7,5.0,0,15,12.039910,1.514138,225
3,1,456,0,3,34,12,1.0965,3.25,1920,53,10,3.5417,7300.0,0.7815,7,7,5.0,0,6,6.799996,0.092123,36
4,1,1568,1,2,31,14,4.5918,3.60,2000,32,12,10.0000,27300.0,0.6215,12,14,9.5,1,7,20.100058,1.524272,49


Utilizando os dados sobre mulheres casadas que trabalham para estimar o retorno da educação no modelo de regressão simples

$ log(wage) = \beta_{0} + \beta_{1}educ + u$

Para comparação, obtendo primeiro as estimativas por MQO

In [4]:
# Removendo linhas com valores ausentes
mroz = mroz.dropna(subset=["lwage", "educ"])

# Variável dependente
y = mroz["lwage"]

# Variáveis independentes
X = mroz[['educ']]

# Adicionando a constante
X = sm.add_constant(X)

# Ajustando o modelo
modelo = sm.OLS(y, X).fit()

# Exibindo o modelo
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.116
Method:                 Least Squares   F-statistic:                     56.93
Date:                Mon, 25 Nov 2024   Prob (F-statistic):           2.76e-13
Time:                        14:58:14   Log-Likelihood:                -441.26
No. Observations:                 428   AIC:                             886.5
Df Residuals:                     426   BIC:                             894.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1852      0.185     -1.000      0.3

O sumário acima, por meio de "edu" indica que 1 ano a mais de educação implica em 10,86% no aumento dos salários (em média).

Abaixo, utilizando a educação do pai (fatheduc) como instrumento a educação (educ). Para isso, tem-se que sustentar que "fathereduc" é não correlacionada com o erro. O segundo requisito é que "educ" e "fatheduc" sejam correlacionados. Para verificar esas suposições pode-se usar uma regressão simples de "educ" sobre "fatheduc"

In [5]:
# Removendo linhas com valores ausentes
mroz = mroz.dropna(subset=["fatheduc", "educ"])

# Variável dependente
y = mroz["educ"]

# Variáveis independentes
X = mroz[['fatheduc']]

# Adicionando a constante
X = sm.add_constant(X)

# Ajustando o modelo
modelo = sm.OLS(y, X).fit()

# Exibindo o modelo
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                   educ   R-squared:                       0.173
Model:                            OLS   Adj. R-squared:                  0.171
Method:                 Least Squares   F-statistic:                     88.84
Date:                Mon, 25 Nov 2024   Prob (F-statistic):           2.76e-19
Time:                        14:58:14   Log-Likelihood:                -920.02
No. Observations:                 428   AIC:                             1844.
Df Residuals:                     426   BIC:                             1852.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         10.2371      0.276     37.099      0.0

Acima, por meio da estatística t de fatheduc a 9,42 indica que educ e fatheduc têm uma correlação positiva e estatísticamente significante e, como mostra R_2 a educação do pai explica cerca de 17% da variação de educação na amostra. Abaixo, utiliza-se fatheduc como instrumento de educ

In [6]:
# Importa os dados
mroz = woo.dataWoo('mroz')

# Variável dependente
y = mroz['lwage']

# Variável endógena
endog = mroz['educ']

# Variável instrumental
instrument = mroz['fatheduc']

# Variáveis exógenas (e.g., constante)
mroz['const'] = 1  
exog = mroz[['const']]  

# Regressão com variável instrumental
iv_model = IV2SLS(dependent=y, exog=exog, endog=endog, instruments=instrument)
iv_results = iv_model.fit()

# Exibe o modelo
print(iv_results.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                      0.0934
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0913
No. Observations:                 428   F-statistic:                    2.5656
Date:                Mon, Nov 25 2024   P-value (F-stat)                0.1092
Time:                        14:58:15   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          0.4411     0.4643     0.9501     0.3421     -0.4689      1.3511
educ           0.0592     0.0369     1.6017     0.10

### Exemplo 15.2 Estimação do retorno da educação para homens

In [7]:
# Importando os dados
wage2 = woo.dataWoo('wage2')
wage2.head()

,wage,hours,IQ,KWW,educ,exper,tenure,age,married,black,south,urban,sibs,brthord,meduc,feduc,lwage
0,769,40,93,35,12,11,2,31,1,0,0,1,1,2.0,8.0,8.0,6.645091
1,808,50,119,41,18,11,16,37,1,0,0,1,1,NaN,14.0,14.0,6.694562
2,825,40,108,46,14,11,9,33,1,0,0,1,1,2.0,14.0,14.0,6.715384
3,650,40,96,32,12,13,7,32,1,0,0,1,4,3.0,12.0,12.0,6.476973
4,562,40,74,27,11,14,5,34,1,0,0,1,10,6.0,6.0,11.0,6.331502


Os resultados abaixo implicam que cada irmão está associado, na média, com cerca de menos 0,23 anos de educação (Aproximadamente 2 meses e 22 dias a menos)

In [8]:
# Removendo linhas com valores ausentes
wage2 = wage2.dropna(subset=["educ", "sibs"])

# Variável dependente
y = wage2["educ"]

# Variáveis independentes
X = wage2[['sibs']]

# Adicionando a constante
X = sm.add_constant(X)

# Ajustando o modelo
modelo = sm.OLS(y, X).fit()

# Exibindo o modelo
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                   educ   R-squared:                       0.057
Model:                            OLS   Adj. R-squared:                  0.056
Method:                 Least Squares   F-statistic:                     56.67
Date:                Mon, 25 Nov 2024   Prob (F-statistic):           1.22e-13
Time:                        14:58:15   Log-Likelihood:                -2034.4
No. Observations:                 935   AIC:                             4073.
Df Residuals:                     933   BIC:                             4083.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         14.1388      0.113    124.969      0.0

Caso se presuma que irmãos (sibs) é não correlacionado com o termo de erro, o estimador de variável instrumental será consistente. Abaixo, a estimação utilizando irmãos (sibs) como instrumento de educação (educ) para a estimação do retorno da educação para homens.

In [9]:
# Importa os dados
wage2 = woo.dataWoo('wage2')

# Removendo linhas com valores ausentes
wage2 = wage2.dropna(subset=["educ", "educ", "sibs"])

# Variável dependente
y = wage2['lwage']

# Variável endógena
endog = wage2['educ']

# Variável instrumental
instrument = wage2['sibs']

# Variáveis exógenas (e.g., constante)
wage2['const'] = 1  
exog = wage2[['const']]  

# Regressão com variável instrumental
iv_model = IV2SLS(dependent=y, exog=exog, endog=endog, instruments=instrument)
iv_results = iv_model.fit()

# Exibe o modelo
print(iv_results.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                     -0.0092
Estimator:                    IV-2SLS   Adj. R-squared:                -0.0103
No. Observations:                 935   F-statistic:                    24.850
Date:                Mon, Nov 25 2024   P-value (F-stat)                0.0000
Time:                        14:58:15   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          5.1300     0.3304     15.528     0.0000      4.4825      5.7776
educ           0.1224     0.0246     4.9850     0.00

OBS: No método acima, IV-2SLS, o R_quadrado foi negativo, uma explicação do porque isso pode ter acontecido e exibido no exmplo do livro.

### Exemplo 15.3 Estimação do efeito do hábito de fumar sobre o peso do nascimento - Exemplo de instrumento Fraco

O exemplo abaixo ilustra porque deve-se sempre verificar se a variável explicativa endógena é correlacionada com a candidata a VI (Variável Instrumental). A equação abaixo estima o efeito do hábito de fumar sobre o peso dos recém nascidos (log do peso apenas em função da quantidade de maços de cigarros fumados)

$log(bwght) = \beta_{0} + \beta_{1}packs + u$

Uma possível variável instrumental de maços seria o preço médio dos cigarros no estado de residência (cigprice). Neste exemplo será considerado que o preço do cigarro e o erro (u) não são correlacionados. Se cigarros são um produto típico de consumo, a teoria econômica básica sugere que packs e cigprice sejam negativamente correlacionados, de forma que cigprice pode ser usado como uma VI de maços. Para verificar isso, regride-se packs sobre cigprice

In [10]:
# Importando os dados
bwght = woo.dataWoo('bwght')
bwght.head()

,faminc,cigtax,cigprice,bwght,fatheduc,motheduc,parity,male,white,cigs,lbwght,bwghtlbs,packs,lfaminc
0,13.5,16.5,122.300003,109,12.0,12.0,1,1,1,0,4.691348,6.8125,0.0,2.602690
1,7.5,16.5,122.300003,133,6.0,12.0,2,1,0,0,4.890349,8.3125,0.0,2.014903
2,0.5,16.5,122.300003,129,NaN,12.0,2,0,0,0,4.859812,8.0625,0.0,-0.693147
3,15.5,16.5,122.300003,126,12.0,12.0,2,1,0,0,4.836282,7.8750,0.0,2.740840
4,27.5,16.5,122.300003,134,14.0,12.0,2,1,1,0,4.897840,8.3750,0.0,3.314186


In [11]:
# Removendo linhas com valores ausentes
bwght = bwght.dropna(subset=["cigprice", "packs"])

# Variável dependente
y = bwght["packs"]

# Variáveis independentes
X = bwght[['cigprice']]

# Adicionando a constante
X = sm.add_constant(X)

# Ajustando o modelo
modelo = sm.OLS(y, X).fit()

# Exibindo o modelo
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  packs   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.1305
Date:                Mon, 25 Nov 2024   Prob (F-statistic):              0.718
Time:                        14:58:15   Log-Likelihood:                -291.47
No. Observations:                1388   AIC:                             586.9
Df Residuals:                    1386   BIC:                             597.4
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0674      0.103      0.658      0.5

A F-estatística é 0.1305 e o valor de p associado é 0.718, o que é muito alto. Isso sugere que o modelo como um todo não é significativo e não explica a variação na variável dependente de forma significativa. O coeficiente do intercepto é 0.0674, com erro padrão de 0.103. O valor t é 0.658 e o valor de p é 0.511, indicando que o intercepto não é estatisticamente significativo ao nível de 5%. O coeficiente da variável "cigprice" (preço do cigarro) é 0.0003, com erro padrão de 0.001. O valor t é 0.361 e o valor de p é 0.718, o que também indica que o preço do cigarro não tem uma relação estatisticamente significativa com o número de pacotes vendidos/consumidos, ou seja correlação igual a zero. O valor de p alto (0.718) sugere que não há evidência para rejeitar a hipótese nula de que o coeficiente é zero.

Como packs e cigprie não são correlacionados, não deve-se usar cigprice como uma VI de packs. Mas o que acontece se o fizermos ?

In [12]:
# Importa os dados
bwght = woo.dataWoo('bwght')

# Removendo linhas com valores ausentes
bwght = bwght.dropna(subset=["lbwght", "cigprice", "packs"])

# Variável dependente
y = bwght['lbwght']

# Variável endógena
endog = bwght['packs']

# Variável instrumental
instrument = bwght['cigprice']

# Variáveis exógenas (e.g., constante)
bwght['const'] = 1  
exog = bwght[['const']]  

# Regressão com variável instrumental
iv_model = IV2SLS(dependent=y, exog=exog, endog=endog, instruments=instrument)
iv_results = iv_model.fit()

# Exibe o modelo
print(iv_results.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                 lbwght   R-squared:                     -23.230
Estimator:                    IV-2SLS   Adj. R-squared:                -23.248
No. Observations:                1388   F-statistic:                    0.1107
Date:                Mon, Nov 25 2024   P-value (F-stat)                0.7394
Time:                        14:58:15   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          4.4481     0.9387     4.7388     0.0000      2.6084      6.2879
packs          2.9887     8.9832     0.3327     0.73

### Exemplo 15.3 Utilizando a proximidade da faculdade como uma VI da educação - Exemplo com apenas uma variável endógena e um instrumento

Para que a distância até a faculdade (nearc4) seja um instrumento válido para o nível de educação (educ), é necessário que não esteja correlacionada com o erro da equação salarial e que tenha uma correlação parcial positiva com educ. A regressão de educ em nearc4 e outras variáveis exógenas indica que, em média, pessoas mais próximas da faculdade possuem 0,32 anos a mais de escolaridade (p-valor < 0,0001). Esse resultado sugere que nearc4 pode ser um instrumento válido para educ, desde que não haja correlação entre nearc4 e o termo de erro da equação salarial.


##### Regredindo educ (variável endógena) sobre nearc4 (possível instrumento de educ)

In [34]:
# Importando os dados
card = woo.dataWoo('card')
card.head()

# Removendo os dados faltantes
card = card.dropna(subset=['educ', 'nearc4', 'exper', 'expersq', 'black', 'smsa', 'south', 'smsa66', 'reg662', 'reg663', 'reg664', 'reg665', 'reg666',
                          'reg667', 'reg668', 'reg669'])

# Variável dependente
y = card['educ']

# Variáveis independentes
X = card[['nearc4', 'exper', 'expersq', 'black', 'smsa', 'south', 'smsa66', 'reg662', 'reg663', 'reg664', 'reg665', 'reg666',
                          'reg667', 'reg668', 'reg669']]

# Adicionando a constante
X = sm.add_constant(X)

# Ajustando o modelo
modelo = sm.OLS(y, X).fit()

# Exibindo o modelo
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                   educ   R-squared:                       0.477
Model:                            OLS   Adj. R-squared:                  0.474
Method:                 Least Squares   F-statistic:                     182.1
Date:                Mon, 25 Nov 2024   Prob (F-statistic):               0.00
Time:                        15:54:34   Log-Likelihood:                -6258.5
No. Observations:                3010   AIC:                         1.255e+04
Df Residuals:                    2994   BIC:                         1.265e+04
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         16.6383      0.241     69.145      0.0

##### Utilizando a variável instrumental

In [32]:
# Importa os dados
card = woo.dataWoo('card')

# Removendo os dados faltantes
card = card.dropna(subset=['lwage', 'educ', 'nearc4', 'exper', 'expersq', 'black', 'smsa', 'south', 'smsa66', 'reg662', 'reg663', 'reg664', 'reg665', 'reg666',
                          'reg667', 'reg668', 'reg669'])

# Variável dependente
y = card['lwage']

# Variável endógena
endog = card['educ']

# Variável instrumental
instrument = card['nearc4']

# Variáveis exógenas (e.g., constante)
card['const'] = 1  
exog = card[['const', 'exper', 'expersq', 'black', 'smsa', 'south', 'smsa66', 'reg662', 'reg663', 'reg664', 'reg665', 'reg666',
                          'reg667', 'reg668', 'reg669']]  

# Regressão com variável instrumental
iv_model = IV2SLS(dependent=y, exog=exog, endog=endog, instruments=instrument)
iv_results = iv_model.fit()

# Exibe o modelo
print(iv_results.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                      0.2382
Estimator:                    IV-2SLS   Adj. R-squared:                 0.2343
No. Observations:                3010   F-statistic:                    840.83
Date:                Mon, Nov 25 2024   P-value (F-stat)                0.0000
Time:                        15:48:24   Distribution:                 chi2(15)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          3.6662     0.9085     4.0352     0.0001      1.8855      5.4468
exper          0.1083     0.0233     4.6376     0.00

### Retorno de educação para mulheres que trabalham - Uma variável endógena e duas variáveis instrumentais

In [38]:
# Importando os dados
mroz = woo.dataWoo('mroz')
mroz.head()

,inlf,hours,kidslt6,kidsge6,age,educ,wage,repwage,hushrs,husage,huseduc,huswage,faminc,mtr,motheduc,fatheduc,unem,city,exper,nwifeinc,lwage,expersq
0,1,1610,1,0,32,12,3.3540,2.65,2708,34,12,4.0288,16310.0,0.7215,12,7,5.0,0,14,10.910060,1.210154,196
1,1,1656,0,2,30,12,1.3889,2.65,2310,30,9,8.4416,21800.0,0.6615,7,7,11.0,1,5,19.499981,0.328512,25
2,1,1980,1,3,35,12,4.5455,4.04,3072,40,12,3.5807,21040.0,0.6915,12,7,5.0,0,15,12.039910,1.514138,225
3,1,456,0,3,34,12,1.0965,3.25,1920,53,10,3.5417,7300.0,0.7815,7,7,5.0,0,6,6.799996,0.092123,36
4,1,1568,1,2,31,14,4.5918,3.60,2000,32,12,10.0000,27300.0,0.6215,12,14,9.5,1,7,20.100058,1.524272,49


In [41]:
# Removendo os dados faltantes
mroz = mroz.dropna(subset=['lwage', 'educ', 'exper', 'expersq', 'motheduc', 'fatheduc'])

# Variável dependente
y = mroz['lwage']

# Variável endógena
endog = mroz['educ']

# Variável instrumental
instrument = mroz[['motheduc', 'fatheduc']]

# Variáveis exógenas (e.g., constante)
mroz['const'] = 1  
exog = mroz[['const', 'exper', 'expersq']]  # Incluindo a constante

# Regressão com variável instrumental
iv_model = IV2SLS(dependent=y, exog=exog, endog=endog, instruments=instrument)
iv_results = iv_model.fit()

# Exibe o modelo
print(iv_results.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                      0.1357
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1296
No. Observations:                 428   F-statistic:                    18.611
Date:                Mon, Nov 25 2024   P-value (F-stat)                0.0003
Time:                        19:44:10   Distribution:                  chi2(3)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          0.0481     0.4278     0.1124     0.9105     -0.7903      0.8865
exper          0.0442     0.0155     2.8546     0.00